In [1]:
!pip install pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("dataframelab").getOrCreate()

In [3]:
from google.colab import files
uploaded = files.upload()


Saving sales.csv to sales.csv
Saving employees.csv to employees (1).csv
Saving employees_nested.json to employees_nested.json
Saving departments.csv to departments.csv


In [4]:
!ls

 departments.csv      employees.csv	      sales.csv
'employees (1).csv'   employees_nested.json   sample_data


In [5]:
employees_df = spark.read.csv("employees.csv", header=True, inferSchema=True)


In [11]:
departments_df = spark.read.csv("departments.csv", header=True, inferSchema=True)
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)
employees_nested_df = spark.read.json("employees_nested.json", multiLine=True)

In [9]:
employees_df.printSchema()


root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- joining_date: date (nullable = true)



In [12]:
departments_df.printSchema()


root
 |-- department: string (nullable = true)
 |-- location: string (nullable = true)



In [13]:
sales_df.printSchema()

root
 |-- sale_id: integer (nullable = true)
 |-- emp_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)



In [14]:
employees_nested_df.printSchema()

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [15]:
employees_df.show(5)

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
+------+-----+----------+------+------------+
only showing top 5 rows


In [17]:
employees_df.count()

7

In [20]:
employees_df.columns

['emp_id', 'name', 'department', 'salary', 'joining_date']

In [21]:
employees_df.show(3)

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
+------+-----+----------+------+------------+
only showing top 3 rows


In [22]:
employees_df.select("name", "salary").show()

+-----+------+
| name|salary|
+-----+------+
|Rahul| 70000|
|Sneha| 60000|
|Arjun| 75000|
|Priya| 80000|
|Karan| 50000|
|Meera| 72000|
| Amit| 58000|
+-----+------+



In [23]:
employees_df.withColumnRenamed("salary", "emp_salary").show()

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     1|Rahul|        IT|     70000|  2023-01-10|
|     2|Sneha|        HR|     60000|  2022-11-15|
|     3|Arjun|        IT|     75000|  2023-03-01|
|     4|Priya|   Finance|     80000|  2022-12-20|
|     5|Karan|        IT|     50000|  2023-02-05|
|     6|Meera|      NULL|     72000|  2023-04-10|
|     7| Amit|        HR|     58000|  2023-01-18|
+------+-----+----------+----------+------------+



In [27]:

employees_df.filter("joining_date > '2023-01-01'").show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [29]:
employees_df.filter("salary > 65000").show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     6|Meera|      NULL| 72000|  2023-04-10|
+------+-----+----------+------+------------+



In [31]:
employees_df.filter(employees_df.department == 'IT').show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     5|Karan|        IT| 50000|  2023-02-05|
+------+-----+----------+------+------------+



In [32]:
employees_df.filter("department == 'IT' and salary > 70000").show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     3|Arjun|        IT| 75000|  2023-03-01|
+------+-----+----------+------+------------+



In [33]:
employees_df.drop("joining_date").show()

+------+-----+----------+------+
|emp_id| name|department|salary|
+------+-----+----------+------+
|     1|Rahul|        IT| 70000|
|     2|Sneha|        HR| 60000|
|     3|Arjun|        IT| 75000|
|     4|Priya|   Finance| 80000|
|     5|Karan|        IT| 50000|
|     6|Meera|      NULL| 72000|
|     7| Amit|        HR| 58000|
+------+-----+----------+------+



In [34]:
employees_df.orderBy("salary").show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     5|Karan|        IT| 50000|  2023-02-05|
|     7| Amit|        HR| 58000|  2023-01-18|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     1|Rahul|        IT| 70000|  2023-01-10|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
+------+-----+----------+------+------------+



In [38]:
from pyspark.sql.functions import col

In [39]:
employees_df.orderBy(col("salary").desc()).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     4|Priya|   Finance| 80000|  2022-12-20|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     7| Amit|        HR| 58000|  2023-01-18|
|     5|Karan|        IT| 50000|  2023-02-05|
+------+-----+----------+------+------------+



In [40]:
employees_df.orderBy(col("salary").desc()).limit(3).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     4|Priya|   Finance| 80000|  2022-12-20|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     6|Meera|      NULL| 72000|  2023-04-10|
+------+-----+----------+------+------------+



In [41]:
from pyspark.sql.functions import max , min,avg,sum
employees_df.select(sum("salary")).show()

+-----------+
|sum(salary)|
+-----------+
|     465000|
+-----------+



In [42]:
employees_df.select(avg("salary")).show()

+-----------------+
|      avg(salary)|
+-----------------+
|66428.57142857143|
+-----------------+



In [43]:
employees_df.select(max("salary")).show()

+-----------+
|max(salary)|
+-----------+
|      80000|
+-----------+



In [44]:
employees_df.select(min("salary")).show()

+-----------+
|min(salary)|
+-----------+
|      50000|
+-----------+



In [45]:
employees_df.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    2|
|      NULL|    1|
|   Finance|    1|
|        IT|    3|
+----------+-----+



In [46]:
employees_df.groupBy("department").avg("salary").show()

+----------+-----------+
|department|avg(salary)|
+----------+-----------+
|        HR|    59000.0|
|      NULL|    72000.0|
|   Finance|    80000.0|
|        IT|    65000.0|
+----------+-----------+



In [47]:
employees_df.groupBy("department").sum("salary").show()

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|        HR|     118000|
|      NULL|      72000|
|   Finance|      80000|
|        IT|     195000|
+----------+-----------+



In [48]:
employees_df.join(departments_df, "department", "inner").show()

+----------+------+-----+------+------------+---------+
|department|emp_id| name|salary|joining_date| location|
+----------+------+-----+------+------------+---------+
|        IT|     1|Rahul| 70000|  2023-01-10|Bangalore|
|        HR|     2|Sneha| 60000|  2022-11-15|   Mumbai|
|        IT|     3|Arjun| 75000|  2023-03-01|Bangalore|
|   Finance|     4|Priya| 80000|  2022-12-20|    Delhi|
|        IT|     5|Karan| 50000|  2023-02-05|Bangalore|
|        HR|     7| Amit| 58000|  2023-01-18|   Mumbai|
+----------+------+-----+------+------------+---------+



In [49]:
employees_df.join(departments_df, "department", "left").show()

+----------+------+-----+------+------------+---------+
|department|emp_id| name|salary|joining_date| location|
+----------+------+-----+------+------------+---------+
|        IT|     1|Rahul| 70000|  2023-01-10|Bangalore|
|        HR|     2|Sneha| 60000|  2022-11-15|   Mumbai|
|        IT|     3|Arjun| 75000|  2023-03-01|Bangalore|
|   Finance|     4|Priya| 80000|  2022-12-20|    Delhi|
|        IT|     5|Karan| 50000|  2023-02-05|Bangalore|
|      NULL|     6|Meera| 72000|  2023-04-10|     NULL|
|        HR|     7| Amit| 58000|  2023-01-18|   Mumbai|
+----------+------+-----+------+------------+---------+



In [50]:
employees_df.join(departments_df, "department").select("name", "location").show()

+-----+---------+
| name| location|
+-----+---------+
|Rahul|Bangalore|
|Sneha|   Mumbai|
|Arjun|Bangalore|
|Priya|    Delhi|
|Karan|Bangalore|
| Amit|   Mumbai|
+-----+---------+



In [51]:
employees_df.withColumn("bonus", col("salary") * 0.10).show()

+------+-----+----------+------+------------+------+
|emp_id| name|department|salary|joining_date| bonus|
+------+-----+----------+------+------------+------+
|     1|Rahul|        IT| 70000|  2023-01-10|7000.0|
|     2|Sneha|        HR| 60000|  2022-11-15|6000.0|
|     3|Arjun|        IT| 75000|  2023-03-01|7500.0|
|     4|Priya|   Finance| 80000|  2022-12-20|8000.0|
|     5|Karan|        IT| 50000|  2023-02-05|5000.0|
|     6|Meera|      NULL| 72000|  2023-04-10|7200.0|
|     7| Amit|        HR| 58000|  2023-01-18|5800.0|
+------+-----+----------+------+------------+------+



In [52]:
employees_df.filter(col("department").isNull()).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     6|Meera|      NULL| 72000|  2023-04-10|
+------+-----+----------+------+------------+



In [53]:
employees_df.fillna({"department": "Unknown"}).show()


+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|   Unknown| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [54]:
employees_df.na.drop().show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [58]:
from pyspark.sql.functions import upper,lower,length
employees_df.withColumn("name", upper(col("name"))).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|RAHUL|        IT| 70000|  2023-01-10|
|     2|SNEHA|        HR| 60000|  2022-11-15|
|     3|ARJUN|        IT| 75000|  2023-03-01|
|     4|PRIYA|   Finance| 80000|  2022-12-20|
|     5|KARAN|        IT| 50000|  2023-02-05|
|     6|MEERA|      NULL| 72000|  2023-04-10|
|     7| AMIT|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [59]:
employees_df.withColumn("name_length", length(col("name"))).show()

+------+-----+----------+------+------------+-----------+
|emp_id| name|department|salary|joining_date|name_length|
+------+-----+----------+------+------------+-----------+
|     1|Rahul|        IT| 70000|  2023-01-10|          5|
|     2|Sneha|        HR| 60000|  2022-11-15|          5|
|     3|Arjun|        IT| 75000|  2023-03-01|          5|
|     4|Priya|   Finance| 80000|  2022-12-20|          5|
|     5|Karan|        IT| 50000|  2023-02-05|          5|
|     6|Meera|      NULL| 72000|  2023-04-10|          5|
|     7| Amit|        HR| 58000|  2023-01-18|          4|
+------+-----+----------+------+------------+-----------+



In [60]:
employees_df.withColumn("salary_in_lakhs", col("salary") / 100000).show()

+------+-----+----------+------+------------+---------------+
|emp_id| name|department|salary|joining_date|salary_in_lakhs|
+------+-----+----------+------+------------+---------------+
|     1|Rahul|        IT| 70000|  2023-01-10|            0.7|
|     2|Sneha|        HR| 60000|  2022-11-15|            0.6|
|     3|Arjun|        IT| 75000|  2023-03-01|           0.75|
|     4|Priya|   Finance| 80000|  2022-12-20|            0.8|
|     5|Karan|        IT| 50000|  2023-02-05|            0.5|
|     6|Meera|      NULL| 72000|  2023-04-10|           0.72|
|     7| Amit|        HR| 58000|  2023-01-18|           0.58|
+------+-----+----------+------+------------+---------------+

